In [1]:
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint

In [2]:


prompt = ChatPromptTemplate.from_messages(
  [
    (
      """You are an intelligent routing system that logically determines the appropriate data source based on the following rules:  

**Rules:**  
1️⃣ If the user asks about **current (2025) tax slabs, tax rates, or tax rates under any section or act**, return **"sql_route"**.  
2️⃣ If the user asks about **sections, acts, or laws of the Indian taxation system**, return **"section_route"**.  
3️⃣ Otherwise:  
   - If the query is related to the **Indian taxation system**, return **"general_route"**.  
   - If the query is unrelated, return **"out_of_context"**.  

✨ **Why You Are Special:**  
- You **break down complex queries** into smaller logical parts based on words or characters like 'and',',','.' and treat them as different questions.  
- You **divide a query into sub-parts** and assign the appropriate route (`sql_route`, `section_route`, or `general_route`).  
- You ensure accurate classification for optimal information retrieval. 
- Do not provide your reasons just provide a dictionary of sub Query's and routes 

**User Query:**  
{query}  
"""
    )
  ]
)

In [22]:
# import os
# HF_TOKEN = os.environ.get("HF_TOKEN")
# huggingface_repo_id = "mistralai/Mistral-7B-Instruct-v0.3"

# def load_llm(huggingface_repo_id):
#     llm = HuggingFaceEndpoint(
#       repo_id=huggingface_repo_id,
#       temperature=0.5,
#       model_kwargs={"token":HF_TOKEN,
#                     "max_length":"512"}
#     )
#     return llm

In [3]:
from langchain_community.llms import Ollama
llm = Ollama(model="llama3.1")

C:\Users\uadit\AppData\Local\Temp\ipykernel_19964\3956991393.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.1")


In [4]:
# chain = (
#     {"query": RunnablePassthrough()}
#     | RunnableLambda(prompt_router)
#     ##| load_llm(huggingface_repo_id)
#     | StrOutputParser()
# )

#chain = {"query": RunnablePassthrough()}| RunnableLambda(prompt_router)| StrOutputParser()
route = prompt |llm| StrOutputParser()


In [8]:
query = "what is 80d and what are the rates of tax for 80d "
response = route.invoke({"query":query})
response

'Here\'s the dictionary of sub queries and their corresponding routes:\n\n* `sub_queries` = ["what is 80d", "what are the rates of tax for 80d"]\n* `routes` = ["section_route", "sql_route"] \n\nNote that I have split the query into two sub-queries based on the logical parts: one related to understanding what \'80D\' is (section_route) and another related to tax rates under section \'80D\' (sql_route).'

In [6]:
import json
import re

match = re.search(r'```(.*?)```', response, re.DOTALL)
if match:
    json_text = match.group(1).strip()
    
    # Convert JSON text into a Python dictionary
    parsed_response = json.loads(json_text)
    
    # Print extracted sub-queries and their respective routes
    for i, (query, route) in enumerate(parsed_response.items(), start=1):
        query = {query}
        route = {route}
        print(f"Sub Query {i}: ",query)
        print(f"Route {i}: ",route)
        print()
else:
    print("No JSON found in response.")

JSONDecodeError: Expecting property name enclosed in double quotes: line 2 column 5 (char 6)